# Predicción de Enfermedad Renal Crónica mediante Aprendizaje Automático

## Evaluación Final – Aprendizaje Automático

### Objetivo

El objetivo de este trabajo es aplicar y comparar dos modelos de clasificación, Regresión Logística y Support Vector Machine (SVM), para predecir la presencia de enfermedad renal crónica (CKD) a partir de características clínicas de los pacientes.

Se utilizará el dataset Chronic Kidney Disease del UCI Machine Learning Repository, que contiene variables numéricas y categóricas.

Finalmente, se comparará el desempeño de ambos modelos para determinar cuál resulta más adecuado para este problema de clasificación.

## 1. Carga del dataset

Se carga el archivo que contiene los datos de los pacientes con y sin enfermedad renal crónica.

In [10]:
from google.colab import files

archivo = files.upload()

Saving chronic_kidney_disease_full.arff to chronic_kidney_disease_full (2).arff


## 2. Lectura del dataset

El archivo ARFF contiene la información de 400 pacientes y 24 variables predictoras, además de la variable objetivo `class`.

Los datos se convierten en un DataFrame de Pandas para poder analizarlos y utilizarlos posteriormente en los modelos de clasificación.

In [11]:
import pandas as pd

columnas = [
    'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba',
    'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc',
    'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class'
]

with open('chronic_kidney_disease_full.arff', 'r') as archivo:
    lineas = archivo.readlines()

inicio = next(i for i, linea in enumerate(lineas)
              if linea.strip().lower() == '@data') + 1

datos = []

for linea in lineas[inicio:]:
    valores = linea.strip().split(',')
    valores = [v for v in valores if v != '']

    if len(valores) == 25:
        datos.append(valores)

df = pd.DataFrame(datos, columns=columnas)

df = df.replace('?', pd.NA)

print("Dimensiones:", df.shape)
df.head()

Dimensiones: (400, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,class
0,48,80,1.020,1,0,<NA>,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,<NA>,normal,notpresent,notpresent,<NA>,...,38,6000,<NA>,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,<NA>,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


## 3. Conversión de variables numéricas

Algunas variables fueron leídas inicialmente como texto. Se convierten a formato numérico aquellas que representan mediciones cuantitativas, para poder utilizarlas correctamente en los modelos.

In [12]:
columnas_numericas = [
    'age', 'bp', 'bgr', 'bu', 'sc',
    'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc'
]

for columna in columnas_numericas:
    df[columna] = pd.to_numeric(df[columna], errors='coerce')

df[columnas_numericas].dtypes

,0
age,float64
bp,float64
bgr,float64
bu,float64
sc,float64
sod,float64
pot,float64
hemo,float64
pcv,float64
wc,float64


## 4. Revisión de valores faltantes

Se revisa cuántos valores faltantes presenta cada variable antes de dividir los datos en entrenamiento y prueba.

In [13]:
faltantes = df.isnull().sum()

faltantes[faltantes > 0].sort_values(ascending=False)

,0
rbc,152
rc,131
wc,106
pot,88
sod,87
pcv,71
pc,65
hemo,52
su,49
sg,47


## 5. Separación de variables predictoras y variable objetivo

Se separan las variables que utilizarán los modelos para realizar la predicción (X) de la variable objetivo (y), que indica si el paciente presenta o no enfermedad renal crónica.

In [14]:
X = df.drop('class', axis=1)

y = df['class'].map({
    'ckd': 1,
    'notckd': 0
})

print("Variables predictoras:", X.shape[1])
print("Distribución de la variable objetivo:")
print(y.value_counts())

Variables predictoras: 24
Distribución de la variable objetivo:
class
1    250
0    150
Name: count, dtype: int64


## 6. División en datos de entrenamiento y prueba

Los datos se dividen en dos conjuntos: 80 % para entrenar los modelos y 20 % para evaluar su desempeño con datos que no fueron utilizados durante el entrenamiento.

La división se realiza manteniendo la proporción de pacientes con y sin enfermedad renal crónica.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Registros de entrenamiento:", len(X_train))
print("Registros de prueba:", len(X_test))

Registros de entrenamiento: 320
Registros de prueba: 80


## 7. Tratamiento de valores faltantes

Los valores faltantes de las variables numéricas se reemplazan por la mediana y los de las variables categóricas por la moda.

La mediana y la moda se calculan únicamente con los datos de entrenamiento y luego se aplican también al conjunto de prueba, evitando que el modelo reciba información procedente de los datos utilizados para evaluarlo.

In [16]:
# Hacemos copias para trabajar sin modificar los datos originales
X_train = X_train.copy()
X_test = X_test.copy()

# Identificamos las variables categóricas
columnas_categoricas = [
    columna for columna in X.columns
    if columna not in columnas_numericas
]

# Variables numéricas: reemplazar faltantes por la mediana del entrenamiento
for columna in columnas_numericas:
    mediana = X_train[columna].median()
    X_train[columna] = X_train[columna].fillna(mediana)
    X_test[columna] = X_test[columna].fillna(mediana)

# Variables categóricas: reemplazar faltantes por la moda del entrenamiento
for columna in columnas_categoricas:
    moda = X_train[columna].mode()[0]
    X_train[columna] = X_train[columna].fillna(moda)
    X_test[columna] = X_test[columna].fillna(moda)

print("Faltantes en entrenamiento:", X_train.isnull().sum().sum())
print("Faltantes en prueba:", X_test.isnull().sum().sum())

Faltantes en entrenamiento: 0
Faltantes en prueba: 0


## 8. Conversión de variables categóricas

Las variables categóricas se transforman en variables numéricas mediante codificación one-hot, para que puedan ser utilizadas por los modelos de clasificación.

La transformación se realiza por separado en los conjuntos de entrenamiento y prueba, manteniendo en ambos las mismas columnas.

In [17]:
X_train = pd.get_dummies(
    X_train,
    columns=columnas_categoricas,
    drop_first=True,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=columnas_categoricas,
    drop_first=True,
    dtype=int
)

# Asegurar que entrenamiento y prueba tengan las mismas columnas
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Columnas en entrenamiento:", X_train.shape[1])
print("Columnas en prueba:", X_test.shape[1])

Columnas en entrenamiento: 39
Columnas en prueba: 39


## 9. Escalado de las variables

Las variables predictoras presentan escalas diferentes. Por ejemplo, algunas corresponden a concentraciones o recuentos y otras toman valores mucho menores.

Para evitar que las diferencias de escala influyan en los modelos, se utiliza StandardScaler. El escalador se ajusta únicamente con los datos de entrenamiento y luego se aplica al conjunto de prueba.

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Entrenamiento:", X_train_scaled.shape)
print("Prueba:", X_test_scaled.shape)

Entrenamiento: (320, 39)
Prueba: (80, 39)


## 10. Modelo 1: Regresión Logística

El primer modelo utilizado es la Regresión Logística, un algoritmo de clasificación que permite estimar la probabilidad de pertenecer a una de dos clases.

En este caso, el modelo busca clasificar a cada paciente como:

- 1: con enfermedad renal crónica (CKD)
- 0: sin enfermedad renal crónica (notCKD)

El modelo se entrena con los datos de entrenamiento y posteriormente se utiliza para realizar predicciones sobre los datos de prueba.

In [19]:
from sklearn.linear_model import LogisticRegression

modelo_lr = LogisticRegression(max_iter=1000)

modelo_lr.fit(X_train_scaled, y_train)

pred_lr = modelo_lr.predict(X_test_scaled)

print("Modelo de Regresión Logística entrenado correctamente.")

Modelo de Regresión Logística entrenado correctamente.


## 11. Evaluación de la Regresión Logística

El desempeño del modelo se evalúa sobre los datos de prueba mediante cuatro métricas: exactitud (accuracy), precisión, sensibilidad (recall) y F1-score.

Estas métricas permiten analizar no solo cuántas predicciones fueron correctas, sino también qué tan bien identifica el modelo a los pacientes con enfermedad renal crónica.

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_lr = accuracy_score(y_test, pred_lr)
precision_lr = precision_score(y_test, pred_lr)
recall_lr = recall_score(y_test, pred_lr)
f1_lr = f1_score(y_test, pred_lr)

print("Accuracy:", round(accuracy_lr, 3))
print("Precisión:", round(precision_lr, 3))
print("Recall:", round(recall_lr, 3))
print("F1-score:", round(f1_lr, 3))

Accuracy: 0.988
Precisión: 1.0
Recall: 0.98
F1-score: 0.99


## 12. Modelo 2: Support Vector Machine (SVM)

El segundo modelo utilizado es Support Vector Machine (SVM).

SVM busca encontrar una frontera que permita separar de la mejor manera posible las dos clases. En este caso, se utiliza para clasificar a los pacientes como con enfermedad renal crónica (CKD) o sin enfermedad renal crónica (notCKD).

Se utiliza un kernel lineal y el modelo se entrena con los mismos datos utilizados para entrenar la Regresión Logística.

In [22]:
from sklearn.svm import SVC

modelo_svm = SVC(kernel='linear')

modelo_svm.fit(X_train_scaled, y_train)

pred_svm = modelo_svm.predict(X_test_scaled)

print("Modelo SVM entrenado correctamente.")

Modelo SVM entrenado correctamente.


## 13. Evaluación del modelo SVM

El modelo SVM se evalúa sobre los mismos datos de prueba utilizados para evaluar la Regresión Logística.

Se calculan las mismas métricas: accuracy, precisión, recall y F1-score, lo que permite realizar una comparación directa entre ambos modelos.

In [23]:
accuracy_svm = accuracy_score(y_test, pred_svm)
precision_svm = precision_score(y_test, pred_svm)
recall_svm = recall_score(y_test, pred_svm)
f1_svm = f1_score(y_test, pred_svm)

print("Accuracy:", round(accuracy_svm, 3))
print("Precisión:", round(precision_svm, 3))
print("Recall:", round(recall_svm, 3))
print("F1-score:", round(f1_svm, 3))

Accuracy: 0.975
Precisión: 1.0
Recall: 0.96
F1-score: 0.98


## 14. Comparación de los modelos

Para comparar el desempeño de Regresión Logística y SVM se presentan conjuntamente las métricas obtenidas por ambos modelos sobre el mismo conjunto de prueba.

In [24]:
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'SVM'],
    'Accuracy': [accuracy_lr, accuracy_svm],
    'Precisión': [precision_lr, precision_svm],
    'Recall': [recall_lr, recall_svm],
    'F1-score': [f1_lr, f1_svm]
})

comparacion.round(3)

,Modelo,Accuracy,Precisión,Recall,F1-score
0,Regresión Logística,0.988,1.0,0.98,0.99
1,SVM,0.975,1.0,0.96,0.98


## 15. Interpretación y conclusión

Ambos modelos mostraron un desempeño muy alto para clasificar la presencia de enfermedad renal crónica.

La Regresión Logística obtuvo un accuracy de 98,8 %, una precisión de 100 %, un recall de 98 % y un F1-score de 99 %.

El modelo SVM obtuvo un accuracy de 97,5 %, una precisión de 100 %, un recall de 96 % y un F1-score de 98 %.

Los dos modelos alcanzaron una precisión del 100 %, lo que indica que, en este conjunto de prueba, los pacientes clasificados por los modelos como CKD realmente pertenecían a esa clase.

Sin embargo, la Regresión Logística obtuvo un accuracy, recall y F1-score ligeramente superiores. En particular, su mayor recall indica que logró identificar una proporción mayor de los pacientes que realmente presentaban enfermedad renal crónica.

Por lo tanto, para este dataset y esta división de los datos, la Regresión Logística presentó el mejor desempeño de los dos modelos evaluados.

Estos resultados corresponden a un conjunto de datos de 400 registros y deben interpretarse como una evaluación del desempeño de los modelos sobre este dataset, sin considerarlos una herramienta de diagnóstico clínico.